In [1]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "krupenye2016great")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Krupenye Kano et al 2016 MPI Database.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_csv(complete_path_1)

df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)
# df.columns

In [3]:
study1 = df[['subject','sex',  'species', 'site', 'study1_condition',
       'study1_target_ms', 'study1_distractor_ms', 'study1_dls',
       'study1_firstlook_target', 'study1_firstlook_distractor',
       'study1_firstlook_correct']]
study1 = study1.assign(experiment_name='study_1')
study2 = df[['subject','sex',  'species', 'site','study2_cond', 'study2_target_ms',
       'study2_distractor_ms', 'study2_dls', 'study2_firstlook_target',
       'study2_firstlook_distractor','study2_firstlook_correct']]
study2 = study2.assign(experiment_name='study_2')

In [4]:
study1.columns = study1.columns.str.replace('study1_', '')
study2.columns = study2.columns.str.replace('study2_', '')
study2.rename(columns={"cond": "condition"}, inplace=True)

In [5]:
data_frames=[study1, study2]
for index, x in enumerate(data_frames):
    x.rename(columns={"subject": "participant",
        'sex':'sex_original',
        'species':'species_original'}, inplace=True) ##standardize names for participants
    x['participant'] = x['participant'].str.rstrip() ##remove spaces
    x['study_id']="krupenye2016great"
    data_frames[index]=x
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)

In [6]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
fulldf['participant'] = fulldf['participant'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['participant'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
fulldf= fulldf.merge(apedf,left_on='participant', right_on='name', how='left')

In [7]:
spe_2=[]  
for index, row in fulldf.iterrows():
    if not pd.isna(row['species']):
        spe_2.append(row['species'])
    else:
        spe_2.append(row['species_original'])
fulldf = fulldf.assign(species=spe_2)

In [8]:
spe_2=[]  
for index, row in fulldf.iterrows():
    if not pd.isna(row['sex']):
        spe_2.append(row['sex'])
    else:
        spe_2.append(row['sex_original'])
fulldf = fulldf.assign(sex=spe_2)

fulldf['sex'].replace('female', 'f', inplace=True, regex=True)
fulldf['sex'].replace('male', 'm', inplace=True, regex=True)

In [9]:
fulldf.columns = fulldf.columns.str.replace('.', '_', regex=True)
# fulldf.columns

In [10]:
fulldf=fulldf[['study_id','experiment_name','participant', 'sex', 'species', 'site', 'condition',  
       'target_ms', 'distractor_ms', 'dls', 'firstlook_target',
       'firstlook_distractor', 'firstlook_correct' ]]


exp1 = fulldf[fulldf['experiment_name'] == 'study_1']
exp2 = fulldf[fulldf['experiment_name'] == 'study_2']

experiments = [[exp1, 'krupenye2016great_exp1'], ##connects df with name of output dataset
                [ exp2, 'krupenye2016great_exp2']]

for x,y in experiments:
    x = x.dropna(axis=1, how='all')## drop empty rows/columns
    comp_out_path_stand = os.path.join(out_pathway, y+'_standardized.csv')
    x.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)
    ##glossaries
    names = x.columns.tolist()
    df = pd.DataFrame(names)
    df = df.rename(columns={0: "column_name"})
    df["description"] = ""
    studyID_glossary=df[["column_name", "description"]]

    comp_out_path_glossary = os.path.join(out_pathway, y+'_glossary.csv')
    studyID_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)